# Lab 2: Automated Knowledge Graph Extraction (LLMGraphTransformer)
*Moving from manual Cypher engineering to AI-driven ontology extraction and automated pipeline construction.*

---
In Lab 6.02, we manually modeled and populated a University Knowledge Graph using Cypher. In enterprise Knowledge Engineering, manually writing `MERGE` statements does not scale to thousands of unstructured documents. 

Today, we will build an automated ingestion pipeline. By the end of this notebook, you will be able to:
1. Connect to your Neo4j AuraDB instance using LangChain's `Neo4jGraph` wrapper.
2. Use **Large Language Models (LLMs)** as reasoning agents to automatically extract Nodes and Relationships from unstructured text.
3. Ingest those extracted `GraphDocuments` directly into Neo4j.
4. Restrict the LLM's extraction using an explicit **Schema Ontology** to prevent hallucinated relationship types.

### 1. Environment Setup & Authentication
Ensure you have two separate text files in the same folder as this Jupyter Notebook:

1. `neo4j_credentials.txt` (Your downloaded AuraDB credentials)
2. `google_credentials.txt` (Create this file and add one line: `GOOGLE_API_KEY=your_api_key_here`)

In [ ]:
# Install required packages for LLM Graph Extraction
%pip install -q langchain langchain-neo4j langchain-google-genai neo4j pandas python-dotenv

import os
import pandas as pd
from dotenv import load_dotenv
from langchain_neo4j import Neo4jGraph
from langchain_google_genai import ChatGoogleGenerativeAI

print("Packages loaded successfully.")

In [ ]:
print("=== 1. Testing Google AI Studio Connection ===")

# Load the Google credentials file
load_dotenv("google_credentials.txt")

if os.getenv("GOOGLE_API_KEY"):
    print("SUCCESS! Google API Key found.")
    llm = ChatGoogleGenerativeAI(
        temperature=0, 
        model="gemini-3.1-flash-lite", 
        max_retries=1
    )
else:
    print("\nFAILED: GOOGLE_API_KEY not found. Check your google_credentials.txt file.")

In [ ]:
#pip install google-genai
from google import genai

# Initialize the client (ensure your GEMINI_API_KEY environment variable is set, 
client = genai.Client()

# List models using the client
for model in client.models.list():
    # The new SDK structure might return slightly different attribute names,
    # but 'name' remains the standard identifier.
    print(model.name)

In [ ]:
print("=== 2. Testing Neo4j Connection ===")

# Load the Neo4j credentials file
load_dotenv("neo4j_credentials.txt")

URI = os.getenv("NEO4J_URI")
USER = os.getenv("NEO4J_USERNAME")
PWD = os.getenv("NEO4J_PASSWORD")

if URI and URI.startswith("neo4j+s://"):
    URI = URI.replace("neo4j+s://", "neo4j+ssc://")

try:
    graph = Neo4jGraph(url=URI, username=USER, password=PWD, enhanced_schema=True)
    graph.refresh_schema()
    print("SUCCESS! Connected to Neo4j automatically. Schema refreshed.")
except Exception as e:
    print(f"\nCONNECTION FAILED! Please check your neo4j_credentials.txt file.\nError:\n{e}")

### 2. Unstructured Data to Graph Documents
LangChain provides a powerful utility called `LLMGraphTransformer`. It prompts an LLM to act as a Knowledge Engineer: it reads chunks of text, identifies named entities (Nodes), determines how they are connected (Relationships), and formats them into a schema Neo4j understands.

In [ ]:
from langchain_core.documents import Document
from langchain_neo4j import LLMGraphTransformer

# 1. Provide unstructured source text
raw_text = """
The Queen's University Belfast focuses on advanced technology education. 
Dr. Alan is a researcher at The Queen's University Belfast who specializes in Artificial Intelligence. 
Artificial Intelligence is heavily utilized in modern autonomous agents and GraphRAG systems.
"""
documents = [Document(page_content=raw_text)]

# 2. Create the Extraction Agent using our Gemini LLM
transformer = LLMGraphTransformer(llm=llm)

# 3. Process the document
print("Extracting graph from text... (this takes a few seconds)")

# NOTE: If you see a warning about "Direct use of automatic function calling (AFC)", 
# this is a harmless upstream message from the Google GenAI SDK and can be safely ignored.
extracted_graph_docs = transformer.convert_to_graph_documents(documents)

# 4. Inspect the extracted topology
print(f"\nExtracted {len(extracted_graph_docs[0].nodes)} Nodes and {len(extracted_graph_docs[0].relationships)} Relationships:\n")

for node in extracted_graph_docs[0].nodes:
    print(f"Node: {node.id} (Type: {node.type})")
    
for rel in extracted_graph_docs[0].relationships:
    print(f"Edge: ({rel.source.id}) -[{rel.type}]-> ({rel.target.id})")

### 3. Ingesting into Neo4j
Currently, the extracted nodes and edges only exist in your computer's RAM. Let's push them into your live Neo4j AuraDB instance in the cloud.

In [ ]:
# Clear database first to ensure a clean slate
graph.query("MATCH (n) DETACH DELETE n")

# Push the GraphDocuments to Neo4j
print("Pushing to Neo4j AuraDB...")
graph.add_graph_documents(
    extracted_graph_docs,
    baseEntityLabel=True, 
    include_source=True   
)

graph.refresh_schema()
print("Graph successfully pushed to Neo4j!")

*Stop and Verify: Open your Neo4j Aura Workspace in your browser and run `MATCH (n)-[r]->(m) RETURN n, r, m`. You should see the LLM-generated graph visualized perfectly!*

### 4. Advanced: Schema-Constrained Extraction

In the previous step, the LLM was allowed to dynamically invent node labels and relationship types without constraints. Look closely at the printout from the previous cell or your Neo4j browser visualization. 

Because LLMs are non-deterministic, your exact output will vary from your classmates, but you likely noticed:
* **Inconsistent Node Types:** The LLM might have classified `Artificial Intelligence` as a `Technology`, a `Concept`, or a `FieldOfStudy`.
* **Invented Relationships:** It might have connected nodes using varying verbs like `FOCUSES_ON`, `SPECIALIZES_IN`, or `UTILIZES`.
* **The Source Node:** In your Neo4j visualization, notice the central node (the Document source chunk) automatically created by `include_source=True`, which points to every extracted entity via `MENTIONS` relationships.

---

#### The Problem: Ontology Drift and Schema Fragmentation
While unconstrained extraction works for a quick test, it causes massive issues across enterprise datasets:
1. **Unpredictable Vocabulary:** If one document extracts `Technology` and another extracts `Concept` for the exact same entity, your graph becomes fragmented.
2. **Relationship Synonyms:** For research activities, different documents might extract `SPECIALIZES_IN`, `EXPERT_IN`, `WORKS_ON`, or `STUDIES`. 
3. **GraphRAG Failures:** Downstream Text-to-Cypher AI agents cannot reliably traverse paths like `(:Researcher)-[:RESEARCHES]->(:FieldOfStudy)` if the ingestion pipeline generates arbitrary, unpredictable edge names.

---

#### The Solution: Enforcing a Target Ontology
To build an enterprise-grade Knowledge Graph, we restrict `LLMGraphTransformer` to an explicit schema using two parameters:
* `allowed_nodes`: `["University", "Researcher", "FieldOfStudy", "Application", "Concept"]`
* `allowed_relationships`: `["WORKS_AT", "FOCUSES_ON", "RESEARCHES", "APPLIED_TO"]`

Before re-running the extraction, we clear the database using `MATCH (n) DETACH DELETE n` so the new schema-conforming entities do not overlap with the unconstrained graph.

In [ ]:
# 1. Clear the existing graph to prevent overlap
print("Clearing the previous database...")
graph.query("MATCH (n) DETACH DELETE n")

# 2. Configure strict ontology constraints
allowed_nodes = ["University", "Researcher", "FieldOfStudy", "Application", "Concept"]
allowed_relationships = ["WORKS_AT", "FOCUSES_ON", "RESEARCHES", "APPLIED_TO"]

strict_transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=allowed_nodes,
    allowed_relationships=allowed_relationships
)

# 3. Extract conforming entities and edges using the exact same text
print("Extracting with strict schema constraints...")
strict_graph_docs = strict_transformer.convert_to_graph_documents(documents)

print(f"\nExtracted {len(strict_graph_docs[0].nodes)} Nodes and {len(strict_graph_docs[0].relationships)} Relationships:\n")
for rel in strict_graph_docs[0].relationships:
    print(f"({rel.source.id} : {rel.source.type}) -[:{rel.type}]-> ({rel.target.id} : {rel.target.type})")

# 4. Ingest into Neo4j
graph.add_graph_documents(strict_graph_docs, baseEntityLabel=True)
graph.refresh_schema()
print("\nStrict graph ingested and schema refreshed successfully.")

### 5. Hands-On Student Exercises

**Exercise 1: Extracting a Research Abstract**
1. Find a short abstract (3-4 sentences) of a research paper online relating to your interests.
2. Define a strict ontology (`allowed_nodes` and `allowed_relationships`) that makes sense for your abstract.
3. Run the `LLMGraphTransformer` to extract the entities and push them to your AuraDB instance.
4. Write a Cypher query using the `graph.query()` method to retrieve and print all nodes connected to your main topic.

In [ ]:
# 1. Paste your chosen abstract
custom_abstract = """

"""
custom_docs = [Document(page_content=custom_abstract.strip())]

# 2. Define a schema that fits your text
my_nodes = [] # e.g., ["Technology", "System", "Challenge"]
my_rels = []  # e.g., ["BUILT_UPON", "RESOLVES"]

# 3. Extract and Ingest
# TODO: Initialize LLMGraphTransformer using your custom schema,
# clear the old graph, and add your new graph documents to Neo4j.


# 4. Query the results
# TODO: Write a Cypher query to find nodes connected to your main topic
query = """
"""
# display(pd.DataFrame(graph.query(query)))